# 🧪 Deepfake Model Benchmarking Tool
### *Input a Model URL to Verify Performance Metrics*

This notebook allows you to evaluate any Deepfake Detection model by providing a direct download link. It will automatically download the dataset, load the model, and generate a comprehensive accuracy report.

In [ ]:
# --- 1. User Input & Configuration ---
# Option A: Hugging Face Repo ID (e.g., "username/model-name")
HF_REPO_ID = ""  # @param {type:"string"}

# Option B: Direct URL (.keras or .h5 link from GitHub/Drive)
MODEL_URL = ""  # @param {type:"string"}

MODEL_FILENAME = "downloaded_model.keras"


## 🛠️ 2. Environment Setup
Installing dependencies and downloading the test dataset.

In [ ]:
!pip install huggingface_hub
!pip install gdown
!pip install numpy tensorflow matplotlib seaborn scikit-learn
# Download dataset if not already present
import os
if not os.path.exists("~140k-real-and-fake-faces.zip"):
    !curl -L -o ~140k-real-and-fake-faces.zip https://www.kaggle.com/api/v1/datasets/download/xhlulu/140k-real-and-fake-faces
    !unzip -q ~140k-real-and-fake-faces.zip -d dataset/

In [ ]:
# Organize Test Set
import shutil
os.makedirs("dataset/test/FAKE", exist_ok=True)
os.makedirs("dataset/test/REAL", exist_ok=True)
!mv dataset/real_vs_fake/real-vs-fake/test/fake/* dataset/test/FAKE/ 2>/dev/null || true
!mv dataset/real_vs_fake/real-vs-fake/test/real/* dataset/test/REAL/ 2>/dev/null || true
print("✅ Test dataset ready.")

import os
import gdown
from huggingface_hub import hf_hub_download

if HF_REPO_ID:
    print(f"🚀 Fetching model from Hugging Face Hub: {HF_REPO_ID}...")
    MODEL_FILENAME = hf_hub_download(repo_id=HF_REPO_ID, filename="deepfake_detection_model.keras")
    print(f"✅ Model cached at: {MODEL_FILENAME}")
elif MODEL_URL:
    print("📥 Processing Model URL...")
    if "drive.google.com" in MODEL_URL:
        # ... (gdown logic) ...
        file_id = ""
        if "/d/" in MODEL_URL: file_id = MODEL_URL.split("/d/")[1].split("/")[0]
        elif "id=" in MODEL_URL: file_id = MODEL_URL.split("id=")[1].split("&")[0]
        gdown.download(f"https://drive.google.com/uc?id={file_id}", MODEL_FILENAME, quiet=False)
    else:
        !curl -L -o {MODEL_FILENAME} "{MODEL_URL}"
else:
    print("❌ Please provide either an HF_REPO_ID or a MODEL_URL.")

In [ ]:
import requests
if MODEL_URL:
    print(f"Downloading model from {MODEL_URL}...")
    !curl -L -o {MODEL_FILENAME} "{MODEL_URL}"
    print("✅ Download complete.")

## 📊 4. Run Evaluation
Loading the model and calculating accuracy metrics.

In [ ]:
import tensorflow as tf
import numpy as np
import time
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, precision_recall_curve, auc, cohen_kappa_score
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Load Model
try:
    model = tf.keras.models.load_model(MODEL_FILENAME)
    print("✅ Model loaded successfully.")
except Exception as e:
    print(f"❌ Error loading model: {e}")

# 2. Load Test Data
test_ds = tf.keras.preprocessing.image_dataset_from_directory(
    Config.TEST_DIR,
    label_mode="binary",
    image_size=Config.IMAGE_SIZE,
    batch_size=Config.BATCH_SIZE,
    shuffle=False
)

# 3. Generate Predictions & Measure Latency
print("Running inference on test set...")
start_time = time.time()
y_true = np.concatenate([y for x, y in test_ds], axis=0)
y_pred_probs = model.predict(test_ds)
end_time = time.time()

y_pred_binary = (y_pred_probs > 0.5).astype(int)
total_images = len(y_true)
avg_latency = (end_time - start_time) / total_images

# 4. Advanced Metrics
accuracy = np.mean(y_true == y_pred_binary)
roc_auc = roc_auc_score(y_true, y_pred_probs)
kappa = cohen_kappa_score(y_true.flatten(), y_pred_binary.flatten())
cm = confusion_matrix(y_true, y_pred_binary)
tn, fp, fn, tp = cm.ravel()
specificity = tn / (tn + fp)
sensitivity = tp / (tp + fn)

# 5. Visualizations
plt.figure(figsize=(18, 5))

# Plot 1: Confusion Matrix
plt.subplot(1, 3, 1)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["FAKE", "REAL"], yticklabels=["FAKE", "REAL"])
plt.title("Confusion Matrix")
plt.ylabel("Actual")
plt.xlabel("Predicted")

# Plot 2: ROC Curve
plt.subplot(1, 3, 2)
fpr, tpr, _ = roc_curve(y_true, y_pred_probs)
plt.plot(fpr, tpr, color="darkorange", lw=2, label=f"ROC curve (area = {roc_auc:.4f})")
plt.plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--")
plt.title("Receiver Operating Characteristic (ROC)")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc="lower right")

# Plot 3: PR Curve
plt.subplot(1, 3, 3)
precision, recall, _ = precision_recall_curve(y_true, y_pred_probs)
plt.plot(recall, precision, color="blue", lw=2, label=f"PR Area = {auc(recall, precision):.4f}")
plt.title("Precision-Recall Curve")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.legend(loc="lower left")

plt.tight_layout()
plt.show()

# 6. Final Report
print(f"
--- Detailed Technical Stats ---")
print(f"Overall Accuracy:   {accuracy*100:.2f}%")
print(f"ROC-AUC Score:      {roc_auc:.4f}")
print(f"Sensitivity (Recall): {sensitivity:.4f}")
print(f"Specificity:        {specificity:.4f}")
print(f"Cohen Kappa:        {kappa:.4f}")
print(f"Inference Latency:  {avg_latency*1000:.2f} ms/image")
print("
--- Classification Report ---")
print(classification_report(y_true, y_pred_binary, target_names=["FAKE", "REAL"]))